# CampusVoice-RT: 재현 노트북 (Colab, NVIDIA L4)

런타임 → GPU 유형을 **L4**로 설정한 뒤 위에서부터 실행. 모든 베이스라인(B1~B4)을 같은 런타임·같은 VAD 설정으로 측정한다.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/ekgP525/asr-challenge.git
%cd asr-challenge
!pip install -q -U "qwen-asr[vllm]" silero-vad soundfile librosa matplotlib
!python -c "from importlib.metadata import version as v; print('torch', v('torch'), 'vllm', v('vllm'), 'qwen-asr', v('qwen-asr'))"

## Hugging Face 인증

Colab 공용 IP는 익명 요청이 속도 제한(429)에 걸린다. 읽기 전용 토큰을 Colab 보안 비밀(🔑)에 `HF_TOKEN`으로 넣고 노트북 접근을 허용한 뒤 실행.

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!huggingface-cli whoami

## 평가셋 다운로드 (Hugging Face Datasets, 링크 TBD)

In [ ]:
!pip install -q huggingface_hub
# audio.tar.gz = one request (2,125 single files trip the Hub rate limit: 1,000 API calls / 5 min)
!huggingface-cli download ekgP115/campusvoice-rt-eval --repo-type dataset --include manifest.jsonl audio.tar.gz --local-dir data/eval --quiet
!tar -xzf data/eval/audio.tar.gz -C data/eval && rm data/eval/audio.tar.gz
!for c in clean snr20 snr10 snr5 snr0; do echo -n "$c: "; ls data/eval/audio/$c | wc -l; done; wc -l data/eval/manifest.jsonl

## 베이스라인 실행 → 채점

시뮬레이터는 가상 시계를 쓰므로 실행 시간은 모델 연산 시간뿐이다(L4 기준 B1·B3 10분 안팎, B2 약 65분, B4 약 40분). 결과는 Drive에 시스템별로 저장하고, 세션이 끊기면 clone·설치·다운로드 후 이 셀을 다시 실행하면 끝난 시스템은 건너뛴다.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import subprocess, shutil, os
SAVE = '/content/drive/MyDrive/campusvoice-rt'
os.makedirs(SAVE, exist_ok=True)
for sid in ['B1', 'B3', 'B2', 'B4']:
    if os.path.exists(f'{SAVE}/runs/{sid}/latency.jsonl'):
        print(sid, 'already done'); continue
    subprocess.run(['python', 'baseline/run_qwen3_asr.py', '--config', f'baseline/configs/{sid}.json',
                    '--manifest', 'data/eval/manifest.jsonl', '--audio-root', 'data/eval', '--out-dir', f'runs/{sid}'], check=True)
    subprocess.run(['python', 'evaluation/evaluate.py', '--config', f'baseline/configs/{sid}.json', '--manifest', 'data/eval/manifest.jsonl',
                    '--predictions', f'runs/{sid}/predictions.jsonl', '--latency', f'runs/{sid}/latency.jsonl',
                    '--env', f'runs/{sid}/environment.json', '--hardware', 'NVIDIA L4'], check=True)
    shutil.copytree(f'runs/{sid}', f'{SAVE}/runs/{sid}', dirs_exist_ok=True)
    shutil.copy(f'results/{sid}.json', f'{SAVE}/{sid}.json')
    print(sid, 'saved to Drive')

## 측정 흔들림 검증 (B2, 40발화 × 3회)

전체 평가셋은 1회 측정이다. 1회 측정이 3회 중앙값과 얼마나 다른지 수치로 남긴다(약 10분).

In [ ]:
!python baseline/run_qwen3_asr.py --config baseline/configs/B2.json --manifest data/eval/manifest.jsonl --audio-root data/eval --out-dir runs/jitter_B2 --limit 200 --repeats 3
!python evaluation/jitter_check.py --single runs/B2/latency.jsonl --repeated runs/jitter_B2/latency.jsonl --out results/jitter_B2.json
!cp results/jitter_B2.json /content/drive/MyDrive/campusvoice-rt/

## 리더보드

In [ ]:
!python evaluation/make_leaderboard.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/leaderboard.md').read()))
display(Image('results/latency_cer_paths.png'))